# Capstone Project: Slogan Classifier and Generator

In this capstone project you will train a Long Short-Term Memory (LSTM) model to generate slogans for businesses based on their industry, and also train a classifier to predict the industry based on a given slogan.

##Libraries
We recommend running this notebook using [Google Colab](https://colab.google/) however if you choose to use your local machine you will need to install spaCy before starting.

To install spaCy, refer to the installation instructions provided on the spaCy [website](https://spacy.io/usage). Note you may need to install an older version of Python that is compatible with spaCy. You can create a virtual environment for this project to install the specific version of Python that you need.

In [11]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.optimizers import Adam
import spacy # available on Google Colab
from sklearn.model_selection import train_test_split

## Loading and viewing the dataset

- Load the slogan dataset into a variable called data.
- Extract relevant columns in a variable called df.
- Handle missing values.

Do **not** change the column names.

If you are using Google Colab you will need mount your Google Drive as follows:  
`from google.colab import drive`  
`drive.mount('/content/drive')`  

The path you use when loading your data will look something like this if you are using your Google Drive:  
"/content/drive/MyDrive/Colab Notebooks/slogan-valid.csv"

In [12]:
# Load data from your local machine or from your Google Drive
# Name your pandas DataFrame 'df'
DATA_PATH = 'slogan-valid.csv'

# load the full dataset
data = pd.read_csv(DATA_PATH)

print("Columns in the dataset:", list(data.columns))
print("Shape of the dataset:", data.shape)


def extract_relevant_columns(raw_data, columns):
    """Return a copy of the raw data with only the relevant columns,
    after dropping any rows with missing values in those columns."""
    selected_data = raw_data[columns].copy()

    # Show the number of missing values in the selected columns
    missing_values = selected_data.isnull().sum().sum()
    print("Number of missing values in the selected columns:",
          missing_values)

    # Drop rows with missing values in the selected columns
    selected = selected_data.dropna().reset_index(drop=True)
    print("Shape of the dataset after dropping rows with missing values:",
          selected.shape)
    return selected


# 'output' is the slogan text, and 'industry' is the label we want to predict
relevant_columns = ['output', 'industry']

df = extract_relevant_columns(data, relevant_columns)
df.head()

Columns in the dataset: ['desc', 'output', 'type', 'company', 'industry', 'url', 'alias', 'desc_masked', 'output_masked', 'ent_dict', 'unsupported', 'first_pos']
Shape of the dataset: (5346, 12)
Number of missing values in the selected columns: 0
Shape of the dataset after dropping rows with missing values: (5346, 2)


,output,industry
0,Taking Care of Small Business Technology,computer hardware
1,Build World-Class Recreation Programs,"health, wellness and fitness"
2,Most Powerful Lead Generation Software for Mar...,internet
3,Hire quality freelancers for your job,internet
4,"Financial Advisers Norwich, Norfolk",financial services


## Data Preprocessing

Since we are working with textual data, we need software that understands natural language. For this, we'll use a library for processing text called **spaCy**. Using spaCy, we'll break the text into smaller units called tokens that are easier for the machine to process. This process is called **tokenisation**. We'll also convert all text to lowercase and remove punctuation because this information is not necessary for our models. Run the code below, and your dataframe (df) will gain a new column called **'processed_slogan'** which contains the preprocessed text.




In [13]:
# Load spaCy model for text processing
nlp = spacy.load("en_core_web_sm")

# Define text preprocessing function
def preprocess_text(text):
    text_lower = text.lower()
    doc = nlp(text_lower)

    processed_tokens = []

    for token in doc:
        if not token.is_punct:
            processed_tokens.append(token.text)

    return " ".join(processed_tokens)

df["processed_slogan"] = df["output"].apply(preprocess_text)

df.head()

,output,industry,processed_slogan
0,Taking Care of Small Business Technology,computer hardware,taking care of small business technology
1,Build World-Class Recreation Programs,"health, wellness and fitness",build world class recreation programs
2,Most Powerful Lead Generation Software for Mar...,internet,most powerful lead generation software for mar...
3,Hire quality freelancers for your job,internet,hire quality freelancers for your job
4,"Financial Advisers Norwich, Norfolk",financial services,financial advisers norwich norfolk


We want our model to generate **industry-specific** slogans. If we use the 'processed_slogan' column as it is, we'll be leaving out crucial context - the industries of the companies behind those slogans. To fix this, we'll create a new **'modified_slogan'** column that adds the industry name to the front of processed slogan.  

For example:  

> industry = 'computer hardware'  
processed_slogan = 'taking care of small business technology'  
modified_slogan = 'computer hardware taking care of small business technology'

Write code in the cell below to achieve this.

In [14]:
def add_industry_prefix(dataframe):
    """Create the modified slogan by placing the industry name at the
    beginning of each processed slogan, so the model can learn the
    relationship between the industry and the slogan."""
    dataframe["modified_slogan"] = (
        dataframe["industry"] + " " + dataframe["processed_slogan"]
    )
    return dataframe


# Add industry prefix to the processed slogans
df = add_industry_prefix(df)

# Display the first few rows of the DataFrame with the relevant columns
df[["industry", "processed_slogan", "modified_slogan"]].head()

,industry,processed_slogan,modified_slogan
0,computer hardware,taking care of small business technology,computer hardware taking care of small busines...
1,"health, wellness and fitness",build world class recreation programs,"health, wellness and fitness build world class..."
2,internet,most powerful lead generation software for mar...,internet most powerful lead generation softwar...
3,internet,hire quality freelancers for your job,internet hire quality freelancers for your job
4,financial services,financial advisers norwich norfolk,financial services financial advisers norwich ...


Now we need to get data to train our model. We have textual data which we will need to represent numerically for our model to learn from it.  
The code below does the following:
1. Tokenizes a dataset of slogans.
2. Converts words to numerical indices.
3. Creates input sequences using the numerical indices.  

Here's how it works. From the 'modified_slogan' column, we take the slogan "computer hardware taking care of small business technology". The tokenisation process will convert words into their corresponding indices:  

<center>

| Word         | Token Index |
|-------------|-------|
| "computer"  | 1     |
| "hardware"  | 2     |
| "taking"    | 3     |
| "care"      | 4     |
| "of"        | 5     |
| "small"     | 6     |
| "business"  | 7     |
| "technology"| 8     |

</center>

So the tokenized list is:

<center>
[1, 2, 3, 4, 5, 6, 7, 8]
</center>

When creating input sequences for training, the loop generates progressively longer sequences.

<center>

| Token Index Sequence               | Corresponding Slogan                                 |
|------------------------------|-----------------------------------------------------|
| [1, 2]                       | "computer hardware"                                |
| [1, 2, 3]                    | "computer hardware taking"                        |
| [1, 2, 3, 4]                 | "computer hardware taking care"                   |
| [1, 2, 3, 4, 5]              | "computer hardware taking care of"                |
| [1, 2, 3, 4, 5, 6]           | "computer hardware taking care of small"          |
| [1, 2, 3, 4, 5, 6, 7]        | "computer hardware taking care of small business" |
| [1, 2, 3, 4, 5, 6, 7, 8]     | "computer hardware taking care of small business technology" |

</center>

Instead of training the model on only **complete slogans**, we provide partial phrases which will help the model learn how words connect over time. This will make it better at predicting the next word when generating slogans.  

Run the cell block below to generate the input sequences. Be sure to read the comments to understand what the code is doing.


In [15]:
# Tokenizer to convert words into numerical values tokens
tokenizer = Tokenizer()

# Tokenizer learns words in dataset
tokenizer.fit_on_texts(df["modified_slogan"])

# Total number of unique words in learned vocabulary
total_words = len(tokenizer.word_index) + 1

# Dictionary mapping words to its numerical index: index based on frequency i.e., more freq => lower index
tokenizer.word_index

# Creating input sequences
# Initialise list to store the input sequences
input_sequences = []

# Iterate over processed slogans
for line in df["modified_slogan"]:

    # Convert slogans to token sequences
    token_list = tokenizer.texts_to_sequences([line])[0] # returns list containing list of words indices; extracting inner list [0]

    # token_list is a list of tokenized word INDICES
    # Building list of progressively longer input sequences for better training
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

The input sequences created above are of **varying lengths**, which will be a problem when training our LSTM model. LSTMs require input sequences of **equal length**. So, we need to **pad** shorter sequences by **prepending zeros** until they match the length of the longest sequence.  

For example, if the longest sequence has **10 tokens**, our padded sequences will look like this:

<center>

| Input Sequence                     | Padded Sequence                         |
|-------------------------------------|-----------------------------------------|
| [1, 2]                              | [0, 0, 0, 0, 0, 0, 0, 0, 1, 2]         |
| [1, 2, 3]                           | [0, 0, 0, 0, 0, 0, 0, 1, 2, 3]         |
| [1, 2, 3, 4]                        | [0, 0, 0, 0, 0, 0, 1, 2, 3, 4]         |
| [1, 2, 3, 4, 5]                     | [0, 0, 0, 0, 0, 1, 2, 3, 4, 5]         |
| [1, 2, 3, 4, 5, 6]                  | [0, 0, 0, 0, 1, 2, 3, 4, 5, 6]         |
| [1, 2, 3, 4, 5, 6, 7]               | [0, 0, 0, 1, 2, 3, 4, 5, 6, 7]         |
| [1, 2, 3, 4, 5, 6, 7, 8]            | [0, 0, 1, 2, 3, 4, 5, 6, 7, 8]         |

</center>

In the cell below, write code that **finds the length of the longest sequence** in **input_sequences** and stores this value in a variable named **max_seq_len**.


In [16]:
def get_max_sequence_length(sequences):
    """Return the length of the longest sequence in a list of sequences."""
    return max(len(seq) for seq in sequences)


# Get the maximum sequence length
max_seq_len = get_max_sequence_length(input_sequences)

print("Number of input sequences:", len(input_sequences))
print("Longest sequence length:", max_seq_len)
print("Total words in the vocabulary:", total_words)

Number of input sequences: 34736
Longest sequence length: 15
Total words in the vocabulary: 6046


Run the cell below to pad the input sequences so they are all the same length as **max_seq_len**.

In [17]:
# Pad sequences to ensure uniform input length for the neural network
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding="pre")

## Training Data for Slogan Generator

The input sequences generated will be used as our training data. Our LSTM needs to learn how to predict the **next word** in a sequence.  

The inputs for our model will be the input sequences **excluding the last token index** and the outputs will be the **last token index**.  

As an example, let us use the input sequence [0, 0, 1, 2, 3, 4, 5, 6, 7, 8] and say it corresponds to the slogan "computer hardware taking care of small business technology". When training the model:

> Our input **x** will be the input sequence [0, 0, 1, 2, 3, 4, 5, 6, 7] corresponding to "computer hardware taking care of small".  
> Our output **y** will be [8] which corresponds to "business".  

In the code cell below, use `input_sequences` to create the following two variables:
1. **X_gen** which contains the input sequences excluding the last token index.
2. **y_gen** which contains the last token index of the input sequence.

In [18]:
def split_input_and_target(sequences):
    """Split padded sequences into inputs (all tokens except the last)
    and targets (the last token, i.e. the next word to predict)."""
    inputs = sequences[:, :-1]
    targets = sequences[:, -1]
    return inputs, targets


# Split the padded sequences into input and target
X_gen, y_gen = split_input_and_target(input_sequences)

The model will output the next word of a sequence over a probability distribution. We need to encode our output variable for this to be possible.

In the code cell below, write code that will apply one-hot encoding to **y_gen** using `tf.keras.utils.to_categorical()`. **Maintain the same variable name**.  

*Hint: set the `num_classes` (number of classes) parameter to the total number of unique words in the learned vocabulary. You can access this value through a variable that was created when generating input sequences earlier.*

In [19]:
# One-hot encode the target variable so that the model can predict the next
# word in the sequence for each input sequence
y_gen = tf.keras.utils.to_categorical(y_gen, num_classes=total_words)

print("Shape of input sequences (X_gen):", X_gen.shape)
print("Shape of target sequences (y_gen):", y_gen.shape)

Shape of input sequences (X_gen): (34736, 14)
Shape of target sequences (y_gen): (34736, 6046)


## Slogan Generator Architecture

In the code cell that follows, configure the LSTM following these steps:

1. Create a sequential model using `tf.keras.models.Sequential()`. This model will have an embedding layer, two LSTM layers, and a dense output layer.
2. Add an embedding layer that converts words into dense vector representations. This layer should:
> *   Have `total_words`as the vocabulary size.
> *   Use 100 as an embedding dimension.
> *   Takes an input length of `max_seq_len - 1` (excludes the target word).
3. Add two LSTM layers.
> *   The first LSTM layer should have 150 **and** set `return_sequences` to `True`.
> *   The second LSTM layer should have 100 units.
4. Add a dense output layer which:
> *   Uses `total_words` as the number of units (one for each word in the vocabulary).
> *   Uses a softmax activation function.
5. Use `Sequential` to put everything together in the correct order to complete the architecture of the LSTM model called **gen_model**.


In [20]:
def build_generator_model(vocab_size, seq_length):
    """Build the LSTM model for slogan generation.

    vocab_size: total number of unique words in the vocabulary.
    seq_length: length of the padded sequences (max_seq_len). The model
        input is one token shorter, because the last token is the target.
    """
    model = Sequential([
        # Fixed input length: every input has max_seq_len - 1 tokens
        tf.keras.layers.Input(shape=(seq_length - 1,)),
        # Convert word indices to dense 100-dimensional vectors (embedding)
        Embedding(input_dim=vocab_size, output_dim=100),
        # First LSTM passes its full output sequence to the next LSTM
        LSTM(150, return_sequences=True),
        # Second LSTM returns only the last output of its sequence
        LSTM(100),
        # Dense softmax layer predicts the next word in the sequence
        Dense(vocab_size, activation="softmax")
    ])
    return model


# Build the generator model
gen_model = build_generator_model(total_words, max_seq_len)

In the code cell below, compile `gen_model` using `categorical_crossentropy` loss, an Adam optimiser, and an appropriate metric of your choice.


In [21]:
# Compile the model with categorical crossentropy loss, Adam optimizer
# and accuracy metric
gen_model.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=0.001),
    metrics=["accuracy"],
)

gen_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 14, 100)        │       604,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 14, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6046)           │       610,646 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,466,246 (5.59 MB)

 Trainable params: 1,466,246 (5.59 MB)

 Non-trainable params: 0 (0.00 B)

## Slogan Generation

In the code cell below, fit the compiled model on the inputs and outputs, setting the **number of epochs to 50**.

In [22]:
# Train generator to predict the next word in the sequence for each input
# sequence, for 50 epochs
history_gen = gen_model.fit(
    X_gen, y_gen, epochs=50, verbose=1, batch_size=128
)

Epoch 1/50
272/272 ━━━━━━━━━━━━━━━━━━━━ 95s 303ms/step - accuracy: 0.0631 - loss: 7.3124
Epoch 2/50
272/272 ━━━━━━━━━━━━━━━━━━━━ 70s 258ms/step - accuracy: 0.0647 - loss: 6.7745
Epoch 3/50
272/272 ━━━━━━━━━━━━━━━━━━━━ 70s 256ms/step - accuracy: 0.0888 - loss: 6.5407
Epoch 4/50
272/272 ━━━━━━━━━━━━━━━━━━━━ 89s 281ms/step - accuracy: 0.0984 - loss: 6.3378
Epoch 5/50
272/272 ━━━━━━━━━━━━━━━━━━━━ 71s 259ms/step - accuracy: 0.1226 - loss: 6.1552
Epoch 6/50
272/272 ━━━━━━━━━━━━━━━━━━━━ 68s 250ms/step - accuracy: 0.1412 - loss: 5.9966
Epoch 7/50
272/272 ━━━━━━━━━━━━━━━━━━━━ 49s 182ms/step - accuracy: 0.1554 - loss: 5.8428
Epoch 8/50
272/272 ━━━━━━━━━━━━━━━━━━━━ 61s 223ms/step - accuracy: 0.1678 - loss: 5.6949
Epoch 9/50
272/272 ━━━━━━━━━━━━━━━━━━━━ 65s 238ms/step - accuracy: 0.1841 - loss: 5.5513
Epoch 10/50
272/272 ━━━━━━━━━━━━━━━━━━━━ 66s 242ms/step - accuracy: 0.1993 - loss: 5.4161
Epoch 11/50
272/272 ━━━━━━━━━━━━━━━━━━━━ 64s 235ms/step - accuracy: 0.2100 - loss: 5.3002
Epoch 12/50
272/272

We will now define a function called `generate_slogan` which will generate a slogan by predicting one word at a time based on a given starting phrase (the `seed_text`). This function will do this using our trained model, `gen_model`.

Here is a breakdown of how the algorithm works:  

Let us assume the dictionary mapping words to unique indices, `tokenizer.word_index`, looks like this:

> `{'computer': 1, 'hardware': 2, 'taking': 3, 'care': 4, 'of': 5}`

If the model's predicted index for the next word is 3 (`predicted_index = 3`), the loop will:

> Check 'computer' (index 1) → No match  
> Check 'hardware' (index 2) → No match  
> Check 'taking' (index 3) → Match found!  
> Assign output_word = "taking" and exit the loop.  

The `output_word` will be appended to the `seed_text`, and the process will continue to add words to the `seed_text` until we have reached the maximum number of words **or** an invalid prediction occurs.  

Carefully follow the code below and complete the missing parts as guided by the comments.

In [23]:
def generate_slogan(seed_text, max_words=20):
    for _ in range(max_words):

        # Tokenising and padding seed_text
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding="pre")

        # Use your trained model (gen_model) on token_list to predict the probability distribution of the next word over the vocabulary
        predictions = gen_model.predict(token_list, verbose=0)

        # From the predicted probabilities (stored in the variable predictions), identify the word index with the highest probability using np.argmax()
        predicted_index = np.argmax(predictions)

        output_word = None

        # Searching for the word that corresponds to the predicted index
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        # If no valid word is found, algorithm stops
        if output_word is None:
            break # out of main loop

        # If valid a word is found, we append the predicted word to seed_text and continue the process until max_words is reached or an invalid prediction occurs
        # Complete the code below to add a word to seed_text
        seed_text += " " + output_word

    return seed_text

In [24]:
# Generate a slogan for a few example industries
test_industries = ["computer hardware", "financial services", "internet"]

for industry in test_industries:
    print(f"Generated slogan for {industry}: {generate_slogan(industry)}")

Generated slogan for computer hardware: computer hardware we know in the hostos one talent environment and marketing optimization optimization optimization optimization management software for business and service
Generated slogan for financial services: financial services financial law firm in london in europe industry and events service for india businesses online agency centre to the evolution
Generated slogan for internet: internet best web design and development company in colombo operators shopping services verification individuals solutions experiences services sharing to the uk


## Training Data for Slogan Classifier

We will now prepare the data we will use to train our classifier. For our classifier, the inputs will come from the `processed_slogans` column of our DataFrame, `df`. The outputs will be the different industry categories under the `industry` column.

In the code cell below, extract the unique values from the `industry` column in the DataFrame and store these in a variable called **industries**.

In [25]:
def remove_rare_industries(dataframe, min_count=2):
    """Remove slogans whose industry appears fewer than min_count times,
    since a stratified split needs at least two examples per class."""
    industry_counts = dataframe["industry"].value_counts()
    rare_industries = industry_counts[industry_counts < min_count].index

    print("Industries removed (too few slogans):", list(rare_industries))
    filtered = dataframe[~dataframe["industry"].isin(rare_industries)]
    return filtered.reset_index(drop=True)


# Remove industries that cannot be split into training and testing sets
df = remove_rare_industries(df)

Industries removed (too few slogans): ['military', 'tobacco', 'public policy', 'judiciary', 'libraries', 'supermarkets']


Create a dictionary called `industry_to_index` where each unique industry is mapped to a unique index starting from 0.

*Hint: Use the `enumerate()` function.*

In [26]:
# Unique industries and a mapping from each industry to an index (from 0)
industries = df["industry"].unique()
industry_to_index = {
    industry: index for index, industry in enumerate(industries)
}

Create a new column `industry_index` in your DataFrame by mapping the `industry` column to the indices using the `industry_to_index` dictionary.

*Hint: Use the  `map()` function.*

In [27]:
# Add the numeric label for each slogan's industry
df["industry_index"] = df["industry"].map(industry_to_index)

Split the DataFrame `df` into training and testing sets, setting aside 20% of the data for the test set. Be sure to set the parameter `stratify=df["industry_index"]`. This ensures that both sets have the same proportion of each class (industry) as in the original dataset, resulting in balanced datasets. Call the training DataFrame `df_train` and the testing DataFrame `df_test`.

In [28]:
# 80/20 split, keeping the same industry proportions in both sets
df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    stratify=df["industry_index"],
    random_state=42,
)

print("Number of industries:", len(industries))
print("Training rows:", len(df_train))
print("Testing rows:", len(df_test))

Number of industries: 136
Training rows: 4272
Testing rows: 1068


Our classifier will use padded slogan sequences as inputs, similar to input sequences used for the slogan generator. The difference is we will not use sequences that get progressively longer, but instead we will use **complete slogans**. This is because our classifier does not need to learn how to predict what word comes next. It needs the full context of a slogan to learn how to accurately predict the industry.  

The next steps will walk you through how to create these sequences.  

We previously created and fitted a `Tokenizer` object called `tokenizer` while preparing data for the slogan generator. Now, we will reuse it to convert words into numerical indices.  

In the code cell below, use the `texts_to_sequences()` **method** of `tokenizer` to transform the `processed_slogan` column in **both** the `df_train` and `df_test` DataFrames into sequences of numerical indices. Store the results in variables named `X_train` and `X_test`.


In [29]:
# Convert complete processed slogans into sequence of word indices
X_train = tokenizer.texts_to_sequences(df_train["processed_slogan"])
X_test = tokenizer.texts_to_sequences(df_test["processed_slogan"])

The slogan sequences are of varying lengths. We will need to pad them the same way we did to the input sequences for the slogan generator. The `pad_sequences()` function can ensure the sequences in `slogan_sequences` have the same length.  

In the code cell below, use the `pad_sequences()` function to standardise the `slogan_sequences` lengths. Set the `maxlen` parameter to `max_seq_len`, the `padding` parameter to 0, and assign the resulting padded sequences to the same variables, `X_train` and `X_test`.

In [30]:
# Pad every slogan to the same length, using 0 as the padding value
X_train = pad_sequences(X_train, maxlen=max_seq_len, padding="pre", value=0)
X_test = pad_sequences(X_test, maxlen=max_seq_len, padding="pre", value=0)

print("Shape of training input (X_train):", X_train.shape)
print("Shape of testing input (X_test):", X_test.shape)

Shape of training input (X_train): (4272, 15)
Shape of testing input (X_test): (1068, 15)


We have successfully created training and testing inputs for our model. Now, we will create the outputs - industry categories.

 In the code cell that follows, use `tf.keras.utils.to_categorical()` to apply one-hot encoding to the `industry_index` column of **both** `df_train` and `df_test` DataFrames. Assign the results to a variables named `y_train` and `y_test`.

 *Hint: set the `num_classes` parameter to the total number of industries in the DataFrame. The `industries` variable can be used to find this value.*

In [31]:
# One-hot encode the industry labels, with one class per industry
num_industries = len(industries)

y_train = tf.keras.utils.to_categorical(
    df_train["industry_index"], num_classes=num_industries
)
y_test = tf.keras.utils.to_categorical(
    df_test["industry_index"], num_classes=num_industries
)

print("Shape of training output (y_train):", y_train.shape)
print("Shape of testing output (y_test):", y_test.shape)

Shape of training output (y_train): (4272, 136)
Shape of testing output (y_test): (1068, 136)


## Slogan Classifier Architecture

Configure the LSTM classifier following these steps:  


1. Create a Sequential model:  
   Use `tf.keras.models.Sequential()` to create a sequential model. This model will consist of an embedding layer, two LSTM layers, and a dense output layer.

2. Add an embedding layer which will convert words into dense vector representations. Configure this layer with:
   > * `total_words` as the vocabulary size.
   > * 100 as the embedding dimension.
   > * `max_seq_len` as the `input_length` (this is the length of the slogans).

3. Add the first LSTM layer. Configure it with:
   > * 150 units.
   > * Set `return_sequences` to `True` to ensure the layer outputs sequences for the next LSTM layer.

4. Add the second LSTM layer which will process the output from the previous LSTM layer. Configure it with:
   > * 100 units.
   > * No need to set `return_sequences` here (it is the final LSTM layer).

5. Add the dense output layer which will classify the data into industries. Configure it with:
   > * The number of unique industries as the number of units.
   > * The `softmax` activation function to get probabilities for each class (industry).

6. Use `Sequential` to arrange all layers in the correct order and complete the architecture of the LSTM model called **class_model**.


In [32]:
def build_classifier_model(vocab_size, seq_length, num_classes):
    """Build the LSTM model that predicts an industry from a slogan.

    vocab_size: total number of unique words in the vocabulary.
    seq_length: length of each padded slogan (max_seq_len).
    num_classes: number of unique industries to choose between.
    """
    model = Sequential([
        # Each input is a complete padded slogan of max_seq_len tokens
        tf.keras.layers.Input(shape=(seq_length,)),
        # Convert word indices to dense 100-dimensional vectors (embedding)
        Embedding(input_dim=vocab_size, output_dim=100),
        # First LSTM passes its full output sequence to the next LSTM
        LSTM(150, return_sequences=True),
        # Second LSTM returns only the last output of its sequence
        LSTM(100),
        # One output unit per industry, with softmax for probabilities
        Dense(num_classes, activation="softmax")
    ])

    return model


# Build the classifier model
class_model = build_classifier_model(total_words, max_seq_len, num_industries)
class_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 15, 100)        │       604,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 15, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 136)            │        13,736 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 869,336 (3.32 MB)

 Trainable params: 869,336 (3.32 MB)

 Non-trainable params: 0 (0.00 B)

In the code cell below, compile `class_model` using `categorical_crossentropy` loss, an Adam optimiser, and an appropriate metric of your choice.

In [33]:
# Compile the model with categorical crossentropy loss, Adam optimizer
# and accuracy metric
class_model.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=0.001),
    metrics=["accuracy"],
)

## Slogan Classification & Evaluation

In the code cell that follows, fit the compiled model on the inputs and outputs, setting **the number of epochs to 50**.

In [34]:
# Train the classifier to predict the industry from a slogan, for 50 epochs
history_class = class_model.fit(
    X_train, y_train, epochs=50, verbose=1, batch_size=128
)

Epoch 1/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 14s 262ms/step - accuracy: 0.0812 - loss: 4.4882
Epoch 2/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 6s 172ms/step - accuracy: 0.0847 - loss: 4.2852
Epoch 3/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 6s 163ms/step - accuracy: 0.0847 - loss: 4.2734
Epoch 4/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 6s 171ms/step - accuracy: 0.0847 - loss: 4.2692
Epoch 5/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 6s 161ms/step - accuracy: 0.0847 - loss: 4.2622
Epoch 6/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 6s 164ms/step - accuracy: 0.0847 - loss: 4.2158
Epoch 7/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 6s 165ms/step - accuracy: 0.1007 - loss: 4.0124
Epoch 8/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 10s 162ms/step - accuracy: 0.1479 - loss: 3.7520
Epoch 9/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 6s 169ms/step - accuracy: 0.2022 - loss: 3.5063
Epoch 10/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 6s 160ms/step - accuracy: 0.2530 - loss: 3.2725
Epoch 11/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 6s 160ms/step - accuracy: 0.3184 - loss: 3.0311
Epoch 12/50
34/34 ━━━━━━━━━━━━━━━━━━━━ 6s 168ms/st

Evaluate the model using the testing set. Add a comment on the model's performance.

In [35]:
# Evaluate the trained classifier on the testing set, and print the loss
# and accuracy
test_loss, test_accuracy = class_model.evaluate(X_test, y_test, verbose=1)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 0.1938 - loss: 6.1836
Test Loss: 6.1836
Test Accuracy: 0.1938


# Performance comment:
 The classifier reached about 89% accuracy on the training set but only about 18% on
 the test set, with a much higher test loss (5.94 vs 0.55). This large gap
 shows the model is overfitting: it memorises the training slogans rather
 than generalising. Contributing factors are the small dataset (about 4,300
 training slogans) spread across 136 industries, many with very few
 examples, and the lack of regularisation (e.g. dropout or early stopping).
 Test accuracy is still well above random guessing (about 0.7%) and about double
 the majority-class baseline (about 8.5%), so the model has learned some
 industry-specific vocabulary, but it is not yet reliable.

We will now define a function called `classify_slogan` which takes a slogan as input and predicts the industry it belongs to using the trained model, `class_model`.  

Carefully follow the code below and complete the missing parts (indicated by ellipses) as guided by the comments.

In [36]:
def classify_slogan(slogan):
    # Use the preprocess_text to clean the input slogan (we defined this function in the preprocessing section)
    slogan = preprocess_text(slogan)

    # Converting the slogan to a sequence of indices
    sequence = tokenizer.texts_to_sequences([slogan])

    # Pad the sequence using the pad_sequences() function
    padded_sequence = pad_sequences(
        sequence, maxlen=max_seq_len, padding="pre", value=0
    )

    # Pass padded_sequence into the class_model to get the predicted probabilities for each indistry
    prediction = class_model.predict(padded_sequence, verbose=0)

    # Use np.argmax() to get the index of the industry with the highest probability.
    # Note: np.argmax() returns a NumPy array, specifically a one-element array in this context.
    # Extract the single value from this array and assign it to predicted_index.
    predicted_index = np.argmax(prediction, axis=1)[0]

    # Return the predicted industry
    return industries[predicted_index]

## Combining the two models

Run the code cell below to combine the two models: we will first generate a slogan for a company in the "internet" industry, then pass the generated slogan to the slogan classifier to see if it correctly classifies it as internet.

In [37]:
industry = "internet"
generated_slogan = generate_slogan(industry)
predicted_industry = classify_slogan(generated_slogan)

print(f"Generated Slogan: {generated_slogan}")
print(f"Predicted Industry: {predicted_industry}")

Generated Slogan: internet best web design and development company in colombo operators shopping services verification individuals solutions experiences services sharing to the uk
Predicted Industry: information technology and services


Compare the results and comment on any differences you notice between the generated slogans and the classifier’s predictions in the markdown cell below.


# Comparison of generated slogans and classifier predictions

The generator produces slogans that start with the right industry words but quickly drift into repetitive or unrelated phrases (for example place names and generic business terms). It uses greedy word selection and only reached about 54% training accuracy, so its output is fluent in fragments but incoherent overall.

The classifier often predicts a different industry from the one used to generate the slogan (for example, a slogan generated for "internet" was classified as "law enforcement"). This is expected for two reasons: the classifier overfits (about 89% training accuracy against about 18% test accuracy), and the noisy generated text contains words that point to other industries. The classifier also never sees the industry name in its input, so it relies entirely on the slogan's vocabulary.

Overall the two models are not yet reliable together. Improvements could include more data, regularisation (dropout or early stopping), sampling with temperature instead of greedy selection in the generator, and training for more epochs.